# Load data

In [12]:
#load data

import altair as alt
import pandas as pd
from IPython.display import display, HTML

names = pd.read_csv("../dpt2020.csv", sep=";")
names.drop(names[names.preusuel == '_PRENOMS_RARES'].index, inplace=True)
#names.drop(names[names.dpt == 'XX'].index, inplace=True)
names.drop(names[names.annais == 'XXXX'].index, inplace=True)

#grouped by (preusuel, sexe) and sum all other components
#allow to sum the number of births for all dpt
grouped = names.groupby(['preusuel', 'sexe', 'annais'], as_index=False).sum()


print("types : \n", grouped.dtypes)

grouped.head()

types : 
 preusuel      str
sexe        int64
annais        str
dpt           str
nombre      int64
dtype: object


,preusuel,sexe,annais,dpt,nombre
0,AADIL,1,1983,84,3
1,AADIL,1,1992,92,3
2,AAHIL,1,2016,95,3
3,AALIYA,2,2017,75,3
4,AALIYAH,2,2001,92971,9


# Usefull functions

In [21]:
def set_empty_year_to_zero_single_name(data:pd.core.frame.DataFrame, name:str):
    """add rows for genders and years where there is no data (set number to 0).
    modify the initial data"""

    new_data = pd.DataFrame(data)
    for sexe in ['1', '2']:
        for year in range(1900, 2020 + 1):
            str_year = str(year)
            if (data.query("sexe == "+sexe+" and annais == '"+str_year+"'").empty) :
                #print("add data", year, sexe)
                new_data.loc[len(new_data)] = [name, int(sexe), str_year, 'XX', 0]



    #new_data = pd.DataFrame(data)
    return new_data

def normalize_dataset_single_name(data:pd.core.frame.DataFrame):
    """divide by the maximum number per sexe.
    copy the initial data"""
    new_data = data.copy()
    query = new_data.query("sexe==1")
    max_nombre_1 = query.loc[query["nombre"].idxmax()].nombre
    query = new_data.query("sexe==2")
    max_nombre_2 =query.loc[query["nombre"].idxmax()].nombre

    for index, row in new_data.iterrows():
        if row.sexe == 1:
            if max_nombre_1 != 0:
                new_data.loc[index, 'nombre']= new_data.loc[index, 'nombre']*100//max_nombre_1
        elif max_nombre_2 != 0:
            new_data.loc[index, 'nombre']= new_data.loc[index, 'nombre']*100//max_nombre_2

    return new_data


# Charts

In [22]:
single_name = None 
normalized_single_name = None

#compute the tables
prenoms = ['CAMILLE', 'LOU', 'CLAUDE', 'ALIX', 'SACHA', 'GWENAEL', 'DOMINIQUE', 'CLÉMENT', 'ALICE', 'DORIAN', 'ALEX', 'ELISA', 'MAXIME', 'CORINNE']
prenoms.sort()
list_single_names = []
list_normalized_single_names = []
for prenom in prenoms:
    single_name = None 
    normalized_single_name = None
    print("load and process "+prenom)
    single_name = grouped[grouped['preusuel'] == prenom]
    single_name = set_empty_year_to_zero_single_name(single_name, prenom)
    normalized_single_name = normalize_dataset_single_name(single_name)
    list_single_names.append(single_name)
    list_normalized_single_names.append(normalized_single_name)

#concatenate the tables
several_names = list_single_names[0].copy()
normalized_several_names = list_normalized_single_names[0].copy()

for i in range(1, len(list_single_names)):
    single_name = list_single_names[i]
    normalized_single_name = list_normalized_single_names[i]

    #copy rows to several_names
    for index, row in single_name.iterrows():
        several_names.loc[len(several_names)] = row

    for index, row in normalized_single_name.iterrows():
        normalized_several_names.loc[len(normalized_several_names)] = row


#create the selection (and the dropdown)
name_dropdown = alt.binding_select(options=prenoms, name='Select a name: ')
name_selection = alt.selection_point(fields=['preusuel'], bind=name_dropdown, value='CAMILLE')

#create the charts
raw_numbers = alt.Chart(several_names).mark_line().encode(
        x=alt.X('annais:T', title="Year", axis=alt.Axis(tickCount=10)),
        y=alt.Y('nombre:Q', title="Number of births"),
        color=alt.Color('sexe:N', title='Gender',
            legend=alt.Legend(
                labelExpr="datum.value == 1 ? 'Boys' : datum.value == 2 ? 'Girls' : datum.value")
        )
).add_params(
    name_selection
).transform_filter(
    name_selection
    #alt.FieldOneOfPredicate(field='preusuel', oneOf=prenoms[0:1])
).properties(
    title='Number of births per gender',
    width=1000
)

normalized_numbers = alt.Chart(normalized_several_names).mark_line(strokeDash=[3,3]).encode(
        x=alt.X('annais:T', title="Year", axis=alt.Axis(tickCount=10)),
        y=alt.Y('nombre:Q', title=['Percentage of births', '(with respect to the maximum of each gender)']),
        color=alt.Color('sexe:N', title='Gender',  
        legend=alt.Legend(
            labelExpr="datum.value == 1 ? 'Boys' : datum.value == 2 ? 'Girls' : datum.value")
        )
).add_params(
    name_selection
).transform_filter(
    name_selection
    #alt.FieldOneOfPredicate(field='preusuel', oneOf=prenoms[0:1])
).properties(
    title='Percentage per gender',
    width=1000
)


#combine the 2 charts
chart = (raw_numbers & normalized_numbers).properties(
    title=' '
)



display(HTML("""
<style>
form.vega-bindings {
  position: absolute;
  left: 10px;
  top: 10px;
  font-weight: bold;
  color: black;
}
</style>
"""))



#chart = chart.configure_title(fontSize=15, offset=5, orient='top', anchor='middle')
#show the chart
chart

load and process ALEX
load and process ALICE
load and process ALIX
load and process CAMILLE
load and process CLAUDE
load and process CLÉMENT
load and process CORINNE
load and process DOMINIQUE
load and process DORIAN
load and process ELISA
load and process GWENAEL
load and process LOU
load and process MAXIME
load and process SACHA


/tmp/ipykernel_61276/57860865.py:76: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  chart = (raw_numbers & normalized_numbers).properties(


alt.VConcatChart(...)